Find the path of the SSD (Will be useful for running the code on the actual dataset)

In [1]:
from pathlib import Path

volumes = Path("/Volumes")

for drive in volumes.iterdir():
    print(drive)

/Volumes/Expansion
/Volumes/Macintosh HD


Validate folder structure before running the code (Code from ai)

In [2]:
from pathlib import Path
import os
import pandas as pd

project_folder = Path(os.getcwd()).parent
dataset_folder = Path("/Volumes/Expansion/lidc_idri")
metadata = project_folder / "data" / "metadata.csv"

df = pd.read_csv(metadata)

missing_paths = []
valid_paths = []
duplicate_targets = {}

for index, row in df.iterrows():

    if row["Modality"] != "CT":
        continue

    patient_id = str(row["Patient ID"])
    study_uid = str(row["Study Instance UID"])
    series_uid = str(row["Series Instance UID"])

    ct_folder = (
        dataset_folder
        / patient_id
        / study_uid
        / series_uid
    )
    
    if ct_folder.exists():
        valid_paths.append(ct_folder)

        # Track destination conflicts
        if patient_id not in duplicate_targets:
            duplicate_targets[patient_id] = 0

        duplicate_targets[patient_id] += 1

    else:
        missing_paths.append(ct_folder)

print("VALIDATION RESULTS")
print("-" * 40)

print(f"Valid CT folders found: {len(valid_paths)}")
print(f"Missing CT folders: {len(missing_paths)}")

if missing_paths:
    print("\nMissing paths:")
    for path in missing_paths[:10]:
        print(path)

print("\nPatients with multiple CT series:")
for patient_id, count in duplicate_targets.items():
    if count > 1:
        print(f"{patient_id}: {count} CT series")

VALIDATION RESULTS
----------------------------------------
Valid CT folders found: 1018
Missing CT folders: 0

Patients with multiple CT series:
LIDC-IDRI-0132: 2 CT series
LIDC-IDRI-0151: 2 CT series
LIDC-IDRI-0315: 2 CT series
LIDC-IDRI-0332: 2 CT series
LIDC-IDRI-0355: 2 CT series
LIDC-IDRI-0365: 2 CT series
LIDC-IDRI-0442: 2 CT series
LIDC-IDRI-0484: 2 CT series


Deletion of DX scans (Since we are not using them for our model)

Note that in order to also delete CR scans the DX is just replaced with CR

Get the study and series instance ID of the DX scan

In [ ]:
from pathlib import Path
import os
import pandas as pd

project_folder = Path(os.getcwd()).parent
dataset_folder = project_folder / "lidc_idri" # To be changed to the actual dataset path
metadata = project_folder / "data" / "metadata.csv"
df = pd.read_csv(metadata)

deleted = 0
missing = 0

# for loop placeholder
row = df[(df["Patient ID"] == "LIDC-IDRI-0001") & (df["Modality"] == "DX")].iloc[0]

# for loop should start here
# actual thing: for index, row in df.iterrows():
# if row["Modality"] == "DX":
study_instance_uid = row["Study Instance UID"]
series_instance_uid = row["Series Instance UID"]

Get the directory to the patient folder, all folders in the patient folder, DX folder

In [ ]:
from pathlib import Path

patient_id = row["Patient ID"]
patient_folder = dataset_folder / patient_id
dx_folder = patient_folder / study_instance_uid / series_instance_uid

Delete the DX folder

In [ ]:
import shutil

if dx_folder.exists():

    print(f"Deleting {row["Modality"]}: {dx_folder}")

    shutil.rmtree(dx_folder)
    deleted += 1

else:
    missing += 1

# End of for loop

Show deletion result

In [ ]:
print("\nFinished")
print(f"Deleted series: {deleted}")
print(f"Missing series: {missing}")

Sorting of CT folders by series instance uid

In [ ]:
from pathlib import Path
import pandas as pd
import shutil
import os

project_folder = Path(os.getcwd()).parent

dataset_folder = Path("/Volumes/Expansion/lidc_idri")
metadata = project_folder / "data" / "metadata.csv"

df = pd.read_csv(metadata)

for index, row in df.iterrows():
    if row["Modality"] == "CT":
        patient_id = row["Patient ID"]
        patient_folder = dataset_folder / patient_id
        study_instance_uid = row["Study Instance UID"]
        series_instance_uid = row["Series Instance UID"]
        ct_folder = patient_folder / study_instance_uid / series_instance_uid

        shutil.move(ct_folder, patient_folder)
        shutil.rmtree(dataset_folder / patient_id / study_instance_uid)

Ai recommends validating after sorting

In [ ]:
from pathlib import Path
import os

project_folder = Path(os.getcwd()).parent
dataset_folder = project_folder / "lidc_idri"

for patient_folder in dataset_folder.iterdir():

    if not patient_folder.is_dir():
        continue

    dicom_files = list(patient_folder.glob("*.dcm"))

    print(f"{patient_folder.name}: {len(dicom_files)} DICOM files")